# Phase 5: Legal Statute Identification (multi-label)
One template for all sessions; set `MODELS` in cell 2:
- Naveen's account, notebook 1: `["llama"]` (input: notebook cs781_train **Version 4**) -> full + LoRMA, knowledge
  distillation (teacher = llama_full, half-size student with and without the teacher), then partial + adapters.
- Naveen's account, notebook 2: `["decoder", "encdec"]` (inputs: cs781_train **Version 1** + dataset **encdec-ckpt**).
- Friend's account: `["moe"]` (input: the friend's notebook output with the MoE best.pt).

Settings: GPU T4 x2, internet on, Add-ons -> Secrets: HF_TOKEN and GH_TOKEN on. Then Save Version -> Save & Run All
(runs with the tab closed, output kept). If it fails while mounting an input, run the cells interactively instead.

In [ ]:
# 1. setup: code from GitHub
import os, subprocess
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
GH = s.get_secret("GH_TOKEN")
REPO = "/kaggle/working/cs781-capstone"
cmds = ([["git", "clone", "-q", f"https://{GH}@github.com/NaveenOLA13/cs781-capstone.git", REPO]]
        if not os.path.isdir(REPO) else
        [["git", "-C", REPO, "fetch", "-q"], ["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"]])
for c in cmds:
    if subprocess.run(c, capture_output=True).returncode != 0:
        raise RuntimeError("git failed: check the GH_TOKEN secret is attached to this notebook")
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. what this session runs
MODELS = ["llama"]                     # ["llama"] | ["decoder", "encdec"] | ["moe"]
TECHS = {"llama": ("full", "lorma"), "decoder": ("full", "lorma"), "encdec": ("full", "bitfit"),
         "moe": ("full", "lorma")}     # encdec: LoRMA on its RNN runs without cuDNN (very slow) -> BitFit instead
MORE = {"llama": ("partial", "adapter")}   # extra techniques after KD (only if time allows)
KD_MODEL = "llama"                     # knowledge distillation on this model (teacher = <model>_full)
SETTINGS = dict(epochs=2, train_chunks=3, eval_chunks=8, doc_tokens=8192, batch_size=8)
BATCH = {"moe": 4}                     # documents per step (x up to train_chunks chunks); MoE is larger
OUT = "/kaggle/working/lsi_runs"

import sys, glob
sys.path.insert(0, f"{REPO}/phase4")
KEY = {"llama": "llama_85m", "decoder": "decoder_80m", "encdec": "encdec", "moe": "moe"}
def find_ckpt(model):
    hits = sorted(p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True)
                  if KEY[model] in p.lower() and os.path.basename(p) == "best.pt")
    return hits[0] if hits else None
CKPT = {m: find_ckpt(m) for m in MODELS}
print(CKPT)
missing = [m for m, p in CKPT.items() if p is None]
if missing:
    raise FileNotFoundError(f"no checkpoint under /kaggle/input for {missing}: attach the inputs and wait for 100%")

In [ ]:
# 3. the data (downloaded from the course's Hugging Face dataset on first use)
from lsi_data import load_data, summary
data = load_data(log=print)
print(summary(data))

In [ ]:
# 4. self-tests (~3 min); stops the notebook if something is broken
r = subprocess.run(f"cd {REPO}/phase4 && python test_lsi_data.py && python test_lsi_train.py", shell=True,
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "self-test failed: send the output above"

In [ ]:
# 5. fine-tuning: two techniques per model, one per GPU (llama_full also saves its training-set probabilities = KD teacher)
from lsi_grid import run_grid, run_kd, ensemble, compare_all
for m in MODELS:
    run_grid(m, techniques=TECHS[m], ckpt=CKPT[m], out_dir=OUT, save_train_probs=(m == KD_MODEL),
             **dict(SETTINGS, batch_size=BATCH.get(m, SETTINGS["batch_size"])))

In [ ]:
# 6. knowledge distillation: half-size student with the teacher (llama_student_kd) and without it (llama_student)
if KD_MODEL in MODELS:
    run_kd(f"{OUT}/{KD_MODEL}_full", student_layers=0.5, out_dir=OUT)

In [ ]:
# 7. more techniques (if time allows)
for m in MODELS:
    if m in MORE:
        run_grid(m, techniques=MORE[m], ckpt=CKPT[m], out_dir=OUT,
                 **dict(SETTINGS, batch_size=BATCH.get(m, SETTINGS["batch_size"])))

In [ ]:
# 8. all runs + ensemble of this session's runs
compare_all(OUT)
TAG = "_".join(MODELS)
ens = ensemble([OUT], out_dir=f"/kaggle/working/lsi_ensemble_{TAG}")

In [ ]:
# 9. results zip (no .pt files) + Hugging Face archive of the weights (skipped if the token cannot write)
import zipfile
from IPython.display import FileLink, display
name = f"lsi_results_{TAG}.zip"
with zipfile.ZipFile(f"/kaggle/working/{name}", "w", zipfile.ZIP_DEFLATED) as z:
    for base in (OUT, f"/kaggle/working/lsi_ensemble_{TAG}"):
        for root, _, files in os.walk(base):
            for f in files:
                if not f.endswith(".pt"):
                    full = os.path.join(root, f)
                    z.write(full, os.path.relpath(full, "/kaggle/working"))
display(FileLink(name))
try:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    hf_repo = f"{api.whoami()['name']}/cs781-phase5-{TAG}"
    api.create_repo(hf_repo, private=True, exist_ok=True)
    api.upload_folder(folder_path="/kaggle/working", repo_id=hf_repo, allow_patterns=["lsi_runs/**"],
                      ignore_patterns=["**/_*"])
    print("weights archived on", hf_repo)
except Exception as e:  # noqa: BLE001
    print("Hugging Face upload skipped:", type(e).__name__, e)